# 3. Test the detector on the ESP32-S3

The firmware (`firmware/rabbitfish_server`) turns the ESP32-S3 into a small HTTP inference
server on your Wi-Fi: `GET /info` describes the model, `POST /detect` takes a JPEG and returns the
boxes. This notebook talks to that API: it checks the board, runs the bundled sample images and
measures mAP on any labelled split, exactly as the device's web page sends images.

**Needs:** a flashed board on the same network (see `firmware/rabbitfish_server/README.md`),
`numpy`, `opencv-python`, `matplotlib`. No GPU and no ML framework.

In [ ]:
import os, sys
from pathlib import Path

# The notebooks live in notebooks/; the shared code is the rabbitfish package at the repository root.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from rabbitfish.paths import DATASET, SAMPLES, RUNS, MODELS, FIRMWARE, WEIGHTS  # also sets YOLO_CONFIG_DIR

# Quick mode (RABBITFISH_QUICK=1) shrinks every expensive step so the whole notebook runs in a few
# minutes. It is how the notebooks are tested; the numbers it produces are meaningless.
QUICK = os.environ.get("RABBITFISH_QUICK") == "1"
print("repository:", ROOT)
print("dataset:   ", DATASET, "(found)" if (DATASET / "data.yaml").exists() else "(MISSING: see data/README.md)")
print("quick mode:", QUICK)

import json, time, urllib.request

HOST = os.environ.get("RABBITFISH_HOST", "rabbitfish.local")   # or the board's IP address
BASE = f"http://{HOST}"

def http_json(url, data=None):
    req = urllib.request.Request(url, data=data, headers={"Content-Type": "image/jpeg"} if data else {})
    with urllib.request.urlopen(req, timeout=60) as r:
        return json.loads(r.read())

## Step 1: is the board there, and is it healthy?

`self_test_passed` means the board recomputed the image stored in the `.espdl` bit for bit like
esp-ppq's simulator: the model, the int8 kernels and the memory layout are all as expected.

In [ ]:
info = http_json(f"{BASE}/info")
print(json.dumps(info, indent=1))
IN_W, IN_H = info["input"]["width"], info["input"]["height"]
assert info["device"]["self_test_passed"], "self-test failed: the model on the board does not match the export"

## Step 2: detect fish in the sample images

Like the web page, each image is shrunk to fit the model input (320x240 for these 4:3 frames)
and sent as a JPEG, so the board never holds a full-size photo. The boxes come back in the
uploaded image's pixels.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

COLORS = {0: (232, 89, 12), 1: (25, 113, 194)}  # fishbody orange, fishhead blue (RGB), as on the device's page
NAMES = {0: "fishbody", 1: "fishhead"}

def show(img_bgr, boxes=(), classes=(), scores=None, title="", ax=None):
    """Draw xyxy boxes (in img_bgr's pixels) on an image."""
    img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB).copy()
    lw = max(2, img.shape[1] // 400)
    for i, (b, c) in enumerate(zip(boxes, classes)):
        x1, y1, x2, y2 = map(int, b)
        cv2.rectangle(img, (x1, y1), (x2, y2), COLORS[int(c)], lw)
        label = NAMES[int(c)] + (f" {scores[i]:.2f}" if scores is not None else "")
        cv2.putText(img, label, (x1, max(12, y1 - 4)), cv2.FONT_HERSHEY_SIMPLEX, lw / 3, COLORS[int(c)], lw)
    ax = ax or plt.figure(figsize=(8, 6)).gca()
    ax.imshow(img); ax.set_title(title); ax.axis("off")

from rabbitfish.data import images

def detect(img_bgr, conf=0.25, quality=92):
    h0, w0 = img_bgr.shape[:2]
    s = min(1.0, IN_W / w0, IN_H / h0)
    small = cv2.resize(img_bgr, (round(w0 * s), round(h0 * s)), interpolation=cv2.INTER_LINEAR)
    jpeg = cv2.imencode(".jpg", small, [cv2.IMWRITE_JPEG_QUALITY, quality])[1].tobytes()
    t0 = time.time()
    res = http_json(f"{BASE}/detect?conf={conf}&max_det=300", jpeg)
    res["round_trip_ms"] = (time.time() - t0) * 1000
    res["scale"] = s
    return res

samples = images("samples", root=SAMPLES.parent)
fig, axes = plt.subplots(2, 3, figsize=(18, 9))
for ax, p in zip(axes.ravel(), samples[:6]):
    img = cv2.imread(str(p))
    res = detect(img)
    d = res["detections"]
    boxes = np.array([x["box"] for x in d]).reshape(-1, 4) / res["scale"]
    show(img, boxes, [x["class_id"] for x in d], [x["score"] for x in d],
         title=f"{p.name}: {res['timing_ms']['inference']:.0f} ms on device", ax=ax)
plt.tight_layout()

## Step 3: accuracy on a labelled split

`tools/evaluate_device.py` sends every image of a split at conf 0.001 and scores the boxes with
ultralytics' matching and 101-point AP, so the result compares directly with the simulated
numbers in the export's `report.json`. Here on the bundled samples (12 images, so the numbers are
noisy); for the full test split use `--split test` with the dataset in place.

Measured for the committed model: val mAP50-95 0.640 (simulated 0.640), test 0.643, at
952 ms per inference and about 1.07 s per request (`results/esp32/README.md`).

In [ ]:
cmd = [sys.executable, str(FIRMWARE / "tools" / "evaluate_device.py"), "--host", HOST,
       "--root", str(SAMPLES.parent), "--split", "samples", "--out", str(RUNS / "device_eval_samples.json")]
print(" ".join(cmd))
import subprocess
print(subprocess.run(cmd, capture_output=True, text=True).stdout[-1500:])